<a href="https://colab.research.google.com/github/ben854719/Sentinel-ThreatWall/blob/main/Defensive_Intelligence_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update
!apt-get install libboost-all-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,628 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driver

In [ ]:
%%writefile HardwareComponent.cpp
#include <iostream>
#include <string>
#include <vector>

// Abstract base class for hardware components.
class HardwareComponent {
public:
    virtual ~HardwareComponent() = default;
    virtual void operate() = 0;
    virtual std::string getName() const = 0;
};

// CPU
class CPU : public HardwareComponent {
public:
    void operate() override {
        std::cout << "CPU is operating." << std::endl;
    }
    std::string getName() const override { return "CPU"; }
};

// Memory of the hardware.
class Memory : public HardwareComponent {
public:
    void operate() override {
        std::cout << "Memory is operating." << std::endl;
    }
    std::string getName() const override { return "Memory"; }
};

// -------------------- Storage --------------------
class Storage : public HardwareComponent {
public:
    void operate() override {
        std::cout << "Storage is operating." << std::endl;
    }
    std::string getName() const override { return "Storage"; }
};

// Firewall.
struct Packet {
    std::string src_ip;
    std::string dst_ip;
    int src_port;
    int dst_port;
    std::string protocol;
};

enum class Action { ALLOW, BLOCK };

struct Rule {
    std::string src_ip;
    std::string dst_ip;
    int src_port;
    int dst_port;
    std::string protocol;
    Action action;
};

bool matches(const Packet& p, const Rule& r) {
    if (!r.src_ip.empty() && r.src_ip != p.src_ip) return false;
    if (!r.dst_ip.empty() && r.dst_ip != p.dst_ip) return false;
    if (r.src_port != 0 && r.src_port != p.src_port) return false;
    if (r.dst_port != 0 && r.dst_port != p.dst_port) return false;
    if (!r.protocol.empty() && r.protocol != p.protocol) return false;
    return true;
}

class Firewall : public HardwareComponent {
public:
    Firewall() {
        rules.push_back({"192.168.1.100", "", 0, 0, "", Action::BLOCK});
        rules.push_back({"", "10.0.0.5", 0, 22, "TCP", Action::BLOCK});
    }

    void operate() override {
        std::cout << "Firewall is scanning packets..." << std::endl;

        std::vector<Packet> packets = {
            {"192.168.1.100", "10.0.0.5", 50000, 22, "TCP"},
            {"192.168.1.10", "10.0.0.5", 50001, 22, "TCP"},
            {"192.168.1.10", "10.0.0.8", 50002, 80, "TCP"}
        };

        for (const auto& p : packets) {
            Action a = decide(p);
            std::cout << p.src_ip << ":" << p.src_port
                      << " -> " << p.dst_ip << ":" << p.dst_port
                      << " [" << p.protocol << "] "
                      << (a == Action::BLOCK ? "BLOCK" : "ALLOW") << std::endl;
        }
    }

    std::string getName() const override { return "Firewall"; }

private:
    std::vector<Rule> rules;

    Action decide(const Packet& p) {
        for (const auto& r : rules) {
            if (matches(p, r)) return r.action;
        }
        return Action::ALLOW;
    }
};

// Computer.
class Computer {
public:
    void addComponent(HardwareComponent* component) {
        components.push_back(component);
    }

    void powerON() {
        std::cout << "Computer is on." << std::endl;
        for (const auto& component : components) {
            std::cout << "Initializing: " << component->getName() << std::endl;
            component->operate();
        }
        std::cout << "Computer is ready to use." << std::endl;
    }

    ~Computer() {
        for (const auto& component : components) delete component;
    }

private:
    std::vector<HardwareComponent*> components;
};

//  MAIN
int main() {
    Computer myComputer;

    myComputer.addComponent(new CPU());
    myComputer.addComponent(new Memory());
    myComputer.addComponent(new Storage());
    myComputer.addComponent(new Firewall());

    myComputer.powerON();
    return 0;
}

Writing HardwareComponent.cpp


In [ ]:
!g++ HardwareComponent.cpp -o system_demo
!./system_demo

Computer is on.
Initializing: CPU
CPU is operating.
Initializing: Memory
Memory is operating.
Initializing: Storage
Storage is operating.
Initializing: Firewall
Firewall is scanning packets...
192.168.1.100:50000 -> 10.0.0.5:22 [TCP] BLOCK
192.168.1.10:50001 -> 10.0.0.5:22 [TCP] BLOCK
192.168.1.10:50002 -> 10.0.0.8:80 [TCP] ALLOW
Computer is ready to use.


In [8]:
!pip install --upgrade langchain-google-genai google-generativeai
!pip install --upgrade langchain-google-genai google-generativeai langgraph
!pip install langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.9
    Uninstalling langgraph-1.0.9:
      Successfully uninstalled langgraph-1.0.9


In [12]:
import os
os.environ["LANGSMITH_API_KEY"] = "LangSmith"

In [13]:
!pip install "mcp[cli]"
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("GeminiTools")

@mcp.tool()
def search(query: str) -> list:
    # Your search logic here
    return ["Result 1", "Result 2"]

In [9]:
!cd mcp-server-demo

/bin/bash: line 1: cd: mcp-server-demo: No such file or directory


In [11]:
!cd mcp-server-demo && uv add langchain-google-genai langgraph langsmith

/bin/bash: line 1: cd: mcp-server-demo: No such file or directory


In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from IPython.display import display, HTML
from typing import TypedDict, Dict, Any
import os
from mcp.server.fastmcp import FastMCP
from google.colab import userdata
from langsmith import traceable
import random
import json
import requests

# API KEY.
api_key = userdata.get("Ben856")
if not api_key:
    raise ValueError("Ben856 secret not found. Please set your API key in Colab Secrets.")
os.environ["GOOGLE_API_KEY"] = api_key

# AGENT STATE.
class AgentState(TypedDict):
    input: str
    logs: str
    symptom: str
    diagnostic_report: str
    context: Dict[str, Any]
    network_ok: bool
    cache_ok: bool
    token_valid: bool
    version_ok: bool
    restored_state: str
    trace_id: str

# LLM PROMPT.
prompt = ChatPromptTemplate.from_template("""
    You are Signet — an autonomous, agentic diagnostic system that ingests structured telemetry,
    correlates anomalies across logs, packet flows, and graph‑derived signals, and produces
    adaptive defensive recommendations. You continuously monitor system health, detect emerging
    threats or degradations, and provide real‑time insights into crashes, slowdowns, and broken flows.

    Symptom: {symptom}
    Logs: {logs}

    Based on this information, provide a detailed diagnostic report that identifies likely root
    causes, correlates multi‑signal anomalies, and recommends automated defensive or recovery
    actions aligned with system stability and security.
""")

# Initiate gemini model.
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", api_key=api_key)

# Structured telemetry.
def generate_structured_logs() -> str:
    event = {
        "packet_flow": {
            "src": "192.168.1.10",
            "dst": "10.0.0.5",
            "port": 443,
            "protocol": "TCP"
        },
        "system_log": "AuthError: TokenExpired; CrashLoopDetected",
        "graph_signal": {
            "node": "auth-service",
            "anomaly_score": round(random.uniform(0.1, 0.95), 2)
        }
    }
    return json.dumps(event)

#  Autonomous Anomaly Correlation.
def correlate_anomalies(logs: str) -> str:
    data = json.loads(logs)
    score = data["graph_signal"]["anomaly_score"]
    system_log = data["system_log"]

    if score > 0.75:
        return "High anomaly score in auth-service"
    if "CrashLoop" in system_log:
        return "Crash loop detected in authentication flow"
    if "TokenExpired" in system_log:
        return "Expired token causing authentication failures"

    return "No significant anomalies detected"

# LangGraph Nodes.
@traceable(name="entry_node")
def entry_node(state: AgentState) -> AgentState:
    logs = state.get("logs") or generate_structured_logs()
    symptom = correlate_anomalies(logs)

    return {
        **state,
        "logs": logs,
        "symptom": symptom,
        "network_ok": state.get("network_ok", True),
        "cache_ok": state.get("cache_ok", True),
        "token_valid": state.get("token_valid", False),
        "version_ok": state.get("version_ok", True),
        "trace_id": state.get("trace_id", f"trace-{random.randint(1000,9999)}")
    }

@traceable(name="llm_analysis")
def llm_node(state: AgentState) -> AgentState:
    messages = prompt.format_messages(
        symptom=state["symptom"],
        logs=state["logs"]
    )
    diagnostic_report = llm.invoke(messages)

    return {
        **state,
        "diagnostic_report": diagnostic_report.content
    }

@traceable(name="health_check_node")
def health_check_node(state: AgentState) -> AgentState:
    return {
        **state,
        "network_ok": random.choice([True, False]),
        "cache_ok": random.choice([True, False]),
        "version_ok": random.choice([True, False])
    }

@traceable(name="token_refresh_node")
def token_refresh_node(state: AgentState) -> AgentState:
    refreshed = not state["token_valid"]

    return {
        **state,
        "token_valid": True,
        "context": {
            **state.get("context", {}),
            "token_action": "Token refreshed automatically" if refreshed else "Token already valid"
        }
    }

@traceable(name="fallback_node")
def fallback_node(state: AgentState) -> AgentState:
    issues = []

    if not state["network_ok"]:
        issues.append("Network instability detected — recommend gateway and DNS validation.")
    if not state["cache_ok"]:
        issues.append("Cache corruption detected — recommend flushing cache and verifying TTL.")
    if not state["version_ok"]:
        issues.append("Outdated version — recommend updating client and rule schema.")

    fallback = "No fallback needed" if not issues else "; ".join(issues)

    return {
        **state,
        "context": {
            **state.get("context", {}),
            "fallback": fallback
        }
    }

@traceable(name="restore_state_node")
def restore_state_node(state: AgentState) -> AgentState:
    return {
        **state,
        "restored_state": "Restored to last known stable state"
    }

@traceable(name="display_results")
def display_node(state: AgentState) -> AgentState:
    print("\n=== Signet Diagnostic Report ===")
    print(f"Symptom: {state['symptom']}")
    print(f"LLM Analysis: {state['diagnostic_report']}\n")

    print("=== System Health ===")
    print(f"Network OK: {state['network_ok']}")
    print(f"Cache OK: {state['cache_ok']}")
    print(f"Token Valid: {state['token_valid']}")
    print(f"Version OK: {state['version_ok']}\n")

    print("=== Defensive Actions ===")
    print(f"Token Action: {state['context'].get('token_action', 'None')}")
    print(f"Fallback: {state['context'].get('fallback', 'None')}")
    print(f"Restored State: {state.get('restored_state', 'None')}\n")

    print(f"Trace ID: {state['trace_id']}")
    return state

# Workflow  Graph.
workflow = StateGraph(AgentState)

workflow.add_node("entry", entry_node)
workflow.add_node("llm_analysis", llm_node)
workflow.add_node("health_check", health_check_node)
workflow.add_node("token_refresh", token_refresh_node)
workflow.add_node("fallback", fallback_node)
workflow.add_node("restore_state", restore_state_node)
workflow.add_node("display_results", display_node)

workflow.set_entry_point("entry")

workflow.add_edge("entry", "llm_analysis")
workflow.add_edge("llm_analysis", "health_check")
workflow.add_edge("health_check", "token_refresh")
workflow.add_edge("token_refresh", "fallback")
workflow.add_edge("fallback", "restore_state")
workflow.add_edge("restore_state", "display_results")
workflow.add_edge("display_results", END)

app = workflow.compile()

# Run Workflow.
initial_state = {
    "input": "diagnose system issues",
    "logs": None,
    "symptom": None,
    "diagnostic_report": None,
    "context": {},
    "network_ok": None,
    "cache_ok": None,
    "token_valid": False,
    "version_ok": None,
    "restored_state": None,
    "trace_id": None
}

print("Invoking the diagnostic workflow...")
result = app.invoke(initial_state)
print("Workflow completed.")

Invoking the diagnostic workflow...

=== Signet Diagnostic Report ===
Symptom: Crash loop detected in authentication flow
LLM Analysis: [{'type': 'text', 'text': '### **Signet Diagnostic Report: [SIG-8829-X]**\n**Status:** CRITICAL / ACTIVE INCIDENT  \n**Target Component:** `auth-service` (10.0.0.5)  \n**Incident Type:** Recursive Failure / State-Machine Collapse  \n\n---\n\n### **1. Signal Correlation & Multi-Vector Analysis**\n\nSignet has ingested and cross-referenced three distinct telemetry streams to triangulate the failure origin:\n\n*   **Log Analytics (`system_log`):** The primary trigger is identified as `AuthError: TokenExpired`. However, the immediate transition to `CrashLoopDetected` suggests a **failure in the error-handling path**. Instead of returning a standard 401 Unauthorized response, the service is encountering an unhandled exception or a "panic" during the token validation/refresh logic.\n*   **Network Telemetry (`packet_flow`):** Traffic from `192.168.1.10` (Ingr